In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
import os
import yaml
import matplotlib.pyplot as plt
from tslearn import metrics, barycenters
import matplotlib.pyplot as plt


from tslearn.preprocessing import TimeSeriesScalerMeanVariance, \
    TimeSeriesResampler

from scipy.signal import find_peaks
import tardigrade_functions as tg

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from matplotlib.gridspec import GridSpec
from scipy import stats
from umap import UMAP



In [ ]:
# settings for dataset
date = 20251209
home = os.path.expanduser("~")
expID = 'LB016'
inpath = os.path.join(home, f'data/{expID}/out_{date}')
pose_path = os.path.join(home, f'data/{expID}/poseestimation')
pose_file = 'DLC_Resnet50_TardiLocoOct23shuffle1_snapshot_200.csv'

In [ ]:
# load config
config_path = "config.yaml"
config = yaml.safe_load(open(config_path, "r"))

fps = config['video']['fps']

bodyparts = config['analysis']['bodyparts']
limbs = config['analysis']['limbs']
limbs_ord = config['analysis']['limbs_ord']
pairs = config['analysis']['pairs']
bp_pairs = config['analysis']['bp_pairs']

bp_color_dict = config['color']['bp_color']
roi_defined = config['analysis']['roi_defined']

idx = pd.IndexSlice

In [ ]:
Trkangle_all = pd.DataFrame([])
CMSvelo_all = pd.DataFrame([])
CLbp_all = pd.DataFrame([])
swing_all = pd.DataFrame([])
turns_all = pd.DataFrame([])
straight_all = pd.DataFrame([])
lastmaxidx = 0
for f in os.listdir(inpath):
    if expID in f and not '.' in f:

        # load angle of tardigrade track
        fangle = pd.read_csv(os.path.join(inpath,f,f'{f}_trackangle.csv'), index_col=0).fillna(0)
        _ = savgol_filter(fangle, fps, 2, axis=0)
        fangle_smooth = pd.DataFrame(_, columns=fangle.columns, index=fangle.index)
        fangle_smooth.index = pd.MultiIndex.from_product([[f],fangle_smooth.index])
        Trkangle_all = pd.concat([Trkangle_all,fangle_smooth])

        # load cms coordinates
        cms = pd.read_csv(os.path.join(inpath,f,f'{f}_stage.csv'),index_col=0).loc[:,['Xcms','Ycms']]
        cmsvelo = np.sqrt(cms.iloc[:,0].diff(5)**2 + cms.iloc[:,1].diff(5)**2).interpolate(limit_direction='both')/5*fps
        CMSvelo_all = pd.concat([CMSvelo_all, cmsvelo])

        # load leg position on centerline
        f_clbp = pd.read_csv(os.path.join(inpath,f,f'{f}_CLbp.csv'), index_col=0, header=[0,1,2])
        f_clbp_norm = (f_clbp - f_clbp.mean(axis=0)) / f_clbp.std(axis=0)
        f_clbp_x = f_clbp_norm.loc[:,idx[:,:,'x']].droplevel(level=[0,2],axis=1).fillna(0)
        CLbp_all = pd.concat([CLbp_all,f_clbp_x])

        # load bool for swings
        fswing = pd.read_csv(os.path.join(inpath,f,f'{f}_swing.csv'), index_col=0)
        fswing.index = pd.MultiIndex.from_product([[f],fswing.index])
        swing_all = pd.concat([swing_all,fswing])
        # load bool for turns
        fturns = pd.read_csv(os.path.join(inpath,f,f'{f}_turns.csv'), index_col=0)+lastmaxidx
        fturns.index = pd.MultiIndex.from_product([[f],fturns.index])
        turns_all = pd.concat([turns_all,fturns])
        # load bool for straight walks
        fstraight = pd.read_csv(os.path.join(inpath,f,f'{f}_straight.csv'), index_col=0)
        fstraight.index = pd.MultiIndex.from_product([[f],fstraight.index])
        st_offset = pd.DataFrame([False]*(len(f_clbp)-len(fstraight)), columns=fstraight.columns)
        st_offset.index = pd.MultiIndex.from_product([[f],st_offset.index])
        straight_all = pd.concat([straight_all, fstraight, st_offset])

        lastmaxidx += len(f_clbp)

In [ ]:
lastmaxidx = 0
rec_cycinfo = {}
cyc_bouts = pd.Series([])
cyc_rec = pd.Series([])
# split recordings into bins based on swings
for i,rec in enumerate(swing_all.index.get_level_values(0).unique()):
    # get swing onsets
    swing_dict = tg.get_bouts(swing_all.loc[rec][limbs])
    swing_on = {k:[s[0] for s in t] for k,t in swing_dict.items()}
    swing_df = pd.DataFrame().from_dict(swing_on, orient='index').T
    # use the first new swing onset of leg to determine bin end
    cyc = swing_df.min(axis=1).astype(int)
    cyc = pd.concat([pd.Series([0]), cyc, pd.Series(len(swing_all.loc[rec]))])
    cyc_ = cyc + lastmaxidx
    # keep bin duration and bin index for each rec
    cyc_info = cyc_.diff().rename('dur').dropna().reset_index(drop=True).to_frame()
    cyc_info['idx'] = cyc_info.index + len(cyc_bouts) #use +len(cyc_bouts) to have one index for all recs
    rec_cycinfo[rec] = cyc_info
    # keep rec index for bins
    cyc_bouts = pd.concat([cyc_bouts,cyc_[:-1]])
    cyc_rec = pd.concat([cyc_rec,pd.Series(np.full_like(cyc_[:-1], i))])
    lastmaxidx += len(swing_all.loc[rec])

cyc_bouts = cyc_bouts.reset_index(drop=True)
len(cyc_bouts), cyc_bouts.diff().mean(), cyc_bouts.diff().std()

In [ ]:
# set base outpath
outpath = os.path.join(inpath, 'cluster', 'out_cluster_050226')
if not os.path.exists(outpath):
    os.makedirs(outpath)

In [ ]:
# example bins
fig, axs = plt.subplots(1,10, figsize=(20,2), sharex=True)
s = 100
for i,j in enumerate(range(s,s+10)):
    axs[i].imshow(swing_all.iloc[cyc_bouts[j]:cyc_bouts[j+1]][limbs_ord].T, aspect='auto', interpolation='None')
plt.savefig(os.path.join(outpath,'examplecycle.png'), bbox_inches='tight')

### cross-similarity

In [ ]:
# wrangle bins for each data type into lists
X_train = [CLbp_all[limbs].iloc[cyc_bouts[j]:cyc_bouts[j+1]].values for j in range(len(cyc_bouts)-1)] #limbs / frontlegs
swing_train = [swing_all[limbs].iloc[cyc_bouts[j]:cyc_bouts[j+1]].values for j in range(len(cyc_bouts)-1)] #limbs / frontlegs
straight_train = [straight_all.iloc[cyc_bouts[j]:cyc_bouts[j+1]].values for j in range(len(cyc_bouts)-1)]
angle_train = [Trkangle_all.iloc[cyc_bouts[j]:cyc_bouts[j+1]].values for j in range(len(cyc_bouts)-1)]
cmsvelo_train = [CMSvelo_all.iloc[cyc_bouts[j]:cyc_bouts[j+1]].values for j in range(len(cyc_bouts)-1)]
cyc_idx_turns = np.concatenate([np.where((cyc_bouts[:-1]<t).values&(cyc_bouts[1:]>t).values)[0] for t in turns_all.values.flatten()])

cyc_trainidx = np.arange(len(cyc_bouts)-1)#[np.array(legs_in_swing) >= 8]

dur_train = [len(x) for x in X_train]
len(dur_train), np.mean(dur_train), np.std(dur_train)

In [ ]:
# resample into same length
ts_resampler = TimeSeriesResampler(sz=15)
X_train = ts_resampler.fit_transform(X_train)
X_train = TimeSeriesScalerMeanVariance().fit_transform(X_train)

# transform bools for swing and straight accordingly
swing_train_new = ts_resampler.transform(swing_train) #TODO check if necessary when below step
swing_train_new = np.round(swing_train_new).astype(int)
straight_train_new = ts_resampler.transform(straight_train) #TODO check if necessary when below step
straight_train_new = np.round(straight_train_new).astype(int)


In [ ]:
# get mean angle and velocity for each bin
angle_train_new = np.vstack([np.nanmean(v) for v in angle_train]).flatten()
cmsvelo_train_new = np.vstack([np.nanmean(v) for v in cmsvelo_train]).flatten()
cmsvelo_train_new.shape

In [ ]:
# compute cross similarity matrix for different sets of bodyparts, using dynamical time warping
dtw_list = []
dtw_range = []
for r in [(0,8), (0,6)]:
    x_train = X_train[:,:,r[0]:r[1]]
    dtw_dist = metrics.cdist_dtw((x_train), (x_train))
    dtw_list.append(dtw_dist)
    dtw_range.append(r)

In [ ]:
# plot cross-similarity in comparable way between sets
triu_idx = np.triu_indices(dtw_list[0].shape[0])
dtw_list_max = np.max([np.max(d) for d in dtw_list])
dtw_list_diff_max = np.max([np.max(abs(dtw_list[0]-d)) for d in dtw_list])+1

for d,r in zip(dtw_list,dtw_range):
    plt.imshow(d, vmin=0, vmax=dtw_list_max)
    plt.colorbar()
    plt.savefig(os.path.join(outpath,f'dtw_crosssimilaritydistance_{r[0]}-{r[1]}.pdf'), bbox_inches='tight')
    plt.show()

    plt.imshow(dtw_list[0]-d,vmin=0, vmax=dtw_list_diff_max)
    plt.colorbar()
    plt.savefig(os.path.join(outpath,f'dtw_crosssimilaritydistance_{r[0]}-{r[1]}_difffull.pdf'), bbox_inches='tight')
    plt.show()

In [ ]:
# calculate the agreement between pairwise cross-similarity matrices using spearman r
agree_mat = np.zeros((len(dtw_list),len(dtw_list)))
for i in range(len(dtw_list)):
    for j in range(len(dtw_list)):
        dtw_i = dtw_list[i]
        dtw_j = dtw_list[j]
        rho, p = stats.spearmanr(dtw_i[triu_idx], dtw_j[triu_idx])
        agree_mat[i,j]  = rho
# plot agreement
plt.imshow(agree_mat, vmin=0, vmax=1)
plt.colorbar()
plt.savefig(os.path.join(outpath,f'dtw_spearmanr.pdf'), bbox_inches='tight')
np.savetxt(os.path.join(outpath,f'dtw_spearmanr.csv'), agree_mat, delimiter=',')

### clustering

In [ ]:
# define x_train for one set of bodyparts, redo for others
dtw_dist = dtw_list[0] # [1]
dtw_name = dtw_range[0] # [1]
x_train = X_train[:,:,dtw_name[0]:dtw_name[1]]
outpath_leg = os.path.join(outpath,f'legs{dtw_name[0]}-{dtw_name[1]}')
if not os.path.exists(outpath_leg):
    os.makedirs(outpath_leg)

In [ ]:
# embedd bins into lower dim space using umap and similarity matrix as pairwise distance

rng = np.random.default_rng(seed=42) #set random seeds to evaluate randomness in UMAP

# UMAP paramaters
neigh = 30
neg_rate = 30
min_dist = 0
disc_dist = 12
rep_str = 1
# define UMAP
emb = UMAP(metric='precomputed', init='random', n_neighbors=neigh, min_dist=min_dist, 
           negative_sample_rate=neg_rate, 
           disconnection_distance=disc_dist, 
           repulsion_strength=rep_str,
           )

# perform embedding multiple times
Xemb_l = []
while len(Xemb_l) < 6:
    emb.random_state = rng.integers(1000)
    print(emb.random_state)
    Xemb = emb.fit_transform(dtw_dist)
    Xemb_l.append(Xemb)

In [ ]:
# plot UMAP embeddings
fig, axs = plt.subplots(2, len(Xemb_l)//2, figsize=(10,5))
axs = axs.T.flatten()
for i in range(len(Xemb_l)):
    sc = axs[i].scatter(*Xemb_l[i].T, alpha=1, s=1, c=cmsvelo_train_new)
    axs[i].axis('equal')
    plt.colorbar(sc)
plt.savefig(os.path.join(outpath_leg,f'UMAP_neigh{neigh}_mindist{min_dist}_negrate{neg_rate}_discdist{disc_dist}_repstr{rep_str}_legs{dtw_name[0]}-{dtw_name[1]}.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# perform clustering on embeddings
# cluster settings
nclust = 4
metr = 'euclidean'
link = 'ward'

# cluster fr all embeddings
cluster_l = []
silscore_l = []
for i in range(len(Xemb_l)):
    cluster = AgglomerativeClustering(n_clusters=nclust, metric=metr, linkage=link).fit(Xemb_l[i])
    cluster_l.append(cluster)
    silscore_l.append(silhouette_score(Xemb_l[i], cluster.labels_))


# plot clustering 
plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.viridis(np.linspace(0,1,len(np.unique(cluster.labels_)))))
fig, axs = plt.subplots(4, len(Xemb_l)//2, figsize=(10,5), height_ratios=(1,.1,1,.1))
axs = axs.T.flatten()
for i,j in enumerate(range(0,len(Xemb_l)*2,2)):
    for c in np.unique(cluster_l[i].labels_):
        axs[j].scatter(*Xemb_l[i][cluster_l[i].labels_==c].T, alpha=1,s=2, label=c)
    axs[j].axis('equal')
    axs[j+1].text(0,0, f'silscore: {np.round(float(silscore_l[i]),4)}')
    axs[j+1].axis('off')
plt.legend()
plt.savefig(os.path.join(outpath_leg,f'AgglomClust_ncl{nclust}_metric{metr}_link{link}.pdf'), bbox_inches='tight')
plt.show()
plt.rcdefaults()


In [ ]:
# calculate agreement matrix between clusters
agree_mat = np.zeros((len(Xemb_l),len(Xemb_l)))
for i in range(len(Xemb_l)):
    for j in range(len(Xemb_l)):
        # get labels for two clusters
        labels_i = cluster_l[i].labels_.copy()
        labels_j = cluster_l[j].labels_.copy()
        
        # calcualte overlap between clusters to map clusters with same identity
        maparr = np.zeros((len(np.unique(labels_i)),len(np.unique(labels_i))))
        for c in np.unique(labels_i):
            n = np.zeros(len(np.unique(labels_i)))
            c_, n_ = np.unique(labels_j[labels_i==c], return_counts=True) # how many of the rows contain the same labels in j as in i
            n[c_] = n_
            maparr[c] = n/np.sum(n)
        
        # maps clusters with highest overlap, create dict of corresponding clusters
        mapper = {k:np.nan for k in np.unique(labels_i)}
        for c in np.argsort(np.max(maparr, axis=1))[::-1]: # sorts cluster with highest overlap to map first
            m_cand = np.argsort(maparr[c])[::-1] # sorts index of clusters in j, according to overlap with clusters in i
            for m in m_cand:
                if m in mapper.values(): # checks if cluster already assigned
                    continue
                mapper[c] = m # assigns cluster
                break

        if not all([o in np.unique(list(mapper.values())) for o in np.unique(labels_i)]):
            raise ValueError(f"Not all values are assigned properly, current mapping dictionary is {mapper}")
        # rename labels in i
        for c in np.unique(labels_i):
            labels_i[labels_i==c] = -mapper[c] #first to negative to not merge different clusters
        labels_i = abs(labels_i) #revert negative with abs
        
        # after clusters are mapped, calculate agreement
        agree_mat[i,j] = np.sum(labels_i == labels_j)/len(labels_i)

# plot agreement
plt.imshow(agree_mat, vmin=0, vmax=1)
plt.colorbar()
plt.savefig(os.path.join(outpath_leg,f'agreement.pdf'), bbox_inches='tight')
np.savetxt(os.path.join(outpath_leg,f'agreement.csv'), agree_mat, delimiter=',')
print(np.mean(agree_mat), np.std(agree_mat))

In [ ]:
# decide for best embedding
best_embedd = 0
Xemb = Xemb_l[best_embedd]
cluster =  cluster_l[best_embedd]

In [ ]:
np.savetxt(os.path.join(outpath_leg,f'Xemb.csv'), Xemb, delimiter=',')
np.savetxt(os.path.join(outpath_leg,f'clusterlabels.csv'), cluster.labels_, delimiter=',')

In [ ]:
# get barycenter for different clusters
barycent_clust = {}
for c in np.unique(cluster.labels_):
    x_c = X_train[cluster.labels_==c]
    barycent_clust[c] = {}
    for bp in range(X_train.shape[2]):
        barycent_clust[c][bp] = barycenters.softdtw_barycenter(x_c[:,:,bp])

In [ ]:
# plot hildebrandt style heatmap of clusters, after each bin has been time warped to barycenter
fig, axs = plt.subplots(len(np.unique(cluster.labels_)),2, figsize=(6,len(np.unique(cluster.labels_))*2.2), sharey=True)
gridspec = axs.flatten()[0].get_subplotspec().get_gridspec()
subfigs = [fig.add_subfigure(gs) for gs in gridspec]

limbs_ord_num = [6,4,2,0,1,3,5,7]
for c in np.unique(cluster.labels_):
    x_train_c = X_train[cluster.labels_==c]
    bary_c = np.hstack(list(barycent_clust[c].values()))
    x_shift = []
    d_shift = []
    print(len(x_train_c))
    # shift each trace in comparision to barycenter
    for x_ in x_train_c:
        path,d = metrics.dtw_path(bary_c, x_)
        path = np.array(path)
        u,ui = np.unique(path[:,0], return_index=True)
        path = path[ui]
        x_shift.append(x_[path[:,1]])
        d_shift.append(d)
    x_shift = np.stack(x_shift)
    x_shift_mean = np.mean(x_shift, axis=0)
    
    # plot mean
    subfigs[0].suptitle(f'mean')
    axs[c,0].set_title(f'Cluster {c}')
    im = axs[c,0].imshow(x_shift_mean.T, vmin=-1, vmax=1) #[limbs_ord_num]
    axs[c,0].set_yticks(range(x_shift_mean.shape[1]))
    axs[c,0].set_yticklabels(limbs) #limbs_ord
    plt.colorbar(im)
    
    # plot barycenter
    subfigs[1].suptitle(f'barycenter')
    axs[c,1].set_title(f'Cluster {c}')
    im = axs[c,1].imshow(bary_c.T, vmin=-1, vmax=1) #[limbs_ord_num]
    plt.colorbar(im)
plt.savefig(os.path.join(outpath_leg,f'hildeheatmap_c{c}.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# plot the movement wave of each leg for the different clusters
limb_color_list = list(pd.Series(bp_color_dict)[limbs])
for c in np.unique(cluster.labels_):

    rndm = np.random.choice(np.arange(len(cluster.labels_))[cluster.labels_==c], min(500, len(cluster.labels_[cluster.labels_==c])))
    bary_c = np.hstack(list(barycent_clust[c].values()))
    x_rndm = []
    d_rndm = []
    for r in rndm:
        path,d = metrics.dtw_path(bary_c[:,dtw_name[0]:dtw_name[1]], x_train[r])
        path = np.array(path)
        u,ui = np.unique(path[:,0], return_index=True)
        path = path[ui]
        x_rndm.append(X_train[r,path[:,1]])
        d_rndm.append(d)
    
    plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.viridis(np.linspace(0,1,len(rndm))))
    fig = plt.figure(layout="constrained", figsize=(4,10))
    gs = GridSpec(5, 2, figure=fig)
    axs = []
    for i in range(4):
        axs.append(fig.add_subplot(gs[i, 0]))
        axs.append(fig.add_subplot(gs[i, 1]))
    axs.append(fig.add_subplot(gs[i+1, :2]))
    axs = np.array(axs)
    
    plt.suptitle(f'Cluster {c}')
    r_i = np.argsort(d_rndm)[::-1]
    for i in range(X_train.shape[2]):
        for j,r in enumerate(r_i):
            x_r = x_rndm[r][:,i]
            axs[i].plot(x_r, alpha=.1, zorder=j)
        axs[i].plot(barycent_clust[c][i], ls=':', c=limb_color_list[i], zorder=len(rndm)+1, lw=3)
        axs[-1].plot(barycent_clust[c][i], ls=':', c=limb_color_list[i], lw=5)

    plt.savefig(os.path.join(outpath_leg,f'gaitwave_c{c}.pdf'), bbox_inches='tight')
    plt.show()
plt.rcdefaults()

### Statistics

In [ ]:
# create a new dataframe o extract statistics 
turns_train_new = np.zeros(len(cluster.labels_))
turns_train_new[cyc_idx_turns] = 1
label_tracktype = pd.DataFrame([cluster.labels_, cmsvelo_train_new, angle_train_new, np.round(np.mean(straight_train_new, axis=1)).astype(int).flatten(), turns_train_new], index=['label','velocity','angle','straight','turn']).T
label_tracktype_rec = label_tracktype.groupby([cyc_rec.reset_index(drop=True),label_tracktype['label']]).mean() # group by biological replicant and label

In [ ]:
# ANOVA
velosamples = []
anglesamples = []
for name, gr in label_tracktype_rec.groupby('label'):
    if name < 0:
        continue
    velosamples.append(gr['velocity'].dropna())
    anglesamples.append(gr['angle'].dropna())
stats.f_oneway(*velosamples), stats.f_oneway(*anglesamples), 

In [ ]:
# tukey_hsd
velores = stats.tukey_hsd(*velosamples)
velo_sig_comb=np.unique(np.array(np.where(velores.pvalue < .05)).T, axis=0)
velo_sig_combuniq=np.unique((np.sort(velo_sig_comb, axis=1)), axis=0)
velo_pv_combuniq=velores.pvalue[velo_sig_combuniq[:,0],velo_sig_combuniq[:,1]]

angleres = stats.tukey_hsd(*anglesamples)
angle_sig_comb=np.unique(np.array(np.where(angleres.pvalue < .05)).T, axis=0)
angle_sig_combuniq=np.unique((np.sort(angle_sig_comb, axis=1)), axis=0)
angle_pv_combuniq=angleres.pvalue[angle_sig_combuniq[:,0],angle_sig_combuniq[:,1]]
velores.pvalue, angleres.pvalue

In [ ]:
def sigbracket_loc(x0, x1, y, h):
    return (x0,y), (x0,y+h), (x1,y+h), (x1,y)

In [ ]:
# Plot boxplot for cluster (animal mean) for velocity and angle
fig, axs = plt.subplots(2, figsize=(7,7))
for name, gr in label_tracktype_rec.groupby('label'):
    if name < 0:
        continue
    axs[0].boxplot(gr['velocity'].dropna(), positions=[name], showfliers=False)
    axs[0].scatter(np.full_like(gr['velocity'].dropna(),name-.3), gr['velocity'].dropna(), c='k')
    axs[1].boxplot(gr['angle'].dropna(), positions=[name], showfliers=False)
    axs[1].scatter(np.full_like(gr['angle'].dropna(),name-.3), gr['angle'].dropna(), c='k')
for pv in range(len(velo_pv_combuniq)):
    p0,p1,p2,p3 = sigbracket_loc(velo_sig_combuniq[pv][0],velo_sig_combuniq[pv][1], 90+(10*velo_sig_combuniq[pv][0]),5)
    axs[0].plot(*np.array([p0,p1,p2,p3]).T)
for pv in range(len(angle_sig_combuniq)):
    p0,p1,p2,p3 = sigbracket_loc(angle_sig_combuniq[pv][0],angle_sig_combuniq[pv][1], .15+(.1*angle_sig_combuniq[pv][0]),.03)
    axs[1].plot(*np.array([p0,p1,p2,p3]).T)
axs[0].set_ylim(0,100)
axs[1].set_ylim(0,0.4)
plt.savefig(os.path.join(outpath_leg,f'clusterdist_velo-angle.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# plot usage and normalized usage of cluster during turn
counts = pd.DataFrame([], index=np.unique(cluster.labels_))
for name, gr in label_tracktype.groupby('turn'):
    count, bin_edge = np.histogram(gr['label'], bins=len(np.unique(cluster.labels_)), range=(0,max(np.unique(cluster.labels_))+1))
    counts.loc[bin_edge[:-1].astype(int),name] = count
    if name == 0:
        continue

plt.hist(gr['label'], histtype='bar', bins=len(np.unique(cluster.labels_)), range=(0,max(np.unique(cluster.labels_))+1))
plt.savefig(os.path.join(outpath_leg,f'clusteruse_turn.pdf'), bbox_inches='tight')
plt.show()

plt.bar(counts.loc[0:].index, counts.loc[0:,1]/counts.loc[0:,0], .9)
plt.savefig(os.path.join(outpath_leg,f'clusteruse_turn_norm.pdf'), bbox_inches='tight')

In [ ]:
# save cluster labels for each recording
all_label_org = pd.DataFrame([])
for f in os.listdir(inpath):
    if expID in f and '.' not in f:
        rec_label = np.full(len(rec_cycinfo[f]), np.nan)
        rec_idx = [t for t in rec_cycinfo[f]['idx'] if t <= len(cluster.labels_)-1]
        rec_label[:len(rec_idx)] = cluster.labels_[rec_idx]
        rec_label_org = np.concatenate([np.repeat(rec_label[i], rec_cycinfo[f]['dur'][i]) for i in range(len(rec_label))]).astype(int)
        print(f, np.unique(rec_label_org, return_counts=True))
        np.savetxt(os.path.join(outpath_leg,f'{f}_gait.csv'), rec_label_org, delimiter=',', fmt='%i')
        rec_label_org = pd.DataFrame(rec_label_org)
        rec_label_org.index = pd.MultiIndex.from_product([[f],rec_label_org.index])
        all_label_org = pd.concat([all_label_org, rec_label_org])